# **STEP 1: INSTALLATION & SETUP**

In [1]:
print("📦 Installing required packages...")

# Install core dependencies
!pip install -q sentence-transformers faiss-cpu groq streamlit pyngrok pypdf2 pandas numpy scikit-learn
!pip install -q datasets torch transformers

print("✅ All packages installed successfully!")

📦 Installing required packages...
✅ All packages installed successfully!


# **STEP 2: IMPORTS**

In [10]:
import os
import json
import pandas as pd
import numpy as np
import faiss
import pickle
from typing import List, Dict, Tuple, Optional
from sentence_transformers import SentenceTransformer, InputExample, losses, evaluation
from torch.utils.data import DataLoader as TorchDataLoader  # Renamed to avoid conflict
from groq import Groq
import PyPDF2
from sklearn.metrics.pairwise import cosine_similarity
import streamlit as st
from pyngrok import ngrok
import threading
import time

print("Imports completed!")

Imports completed!


# **STEP 3: SAMPLE DATASET**

In [11]:
# Create a sample HR/FAQ dataset for testing
SAMPLE_DATASET = {
    "faqs": [
        {
            "question": "What are the company's working hours?",
            "answer": "Our standard working hours are 9:00 AM to 6:00 PM, Monday through Friday. We also offer flexible working arrangements."
        },
        {
            "question": "How many vacation days do employees get?",
            "answer": "Full-time employees receive 20 vacation days per year, plus 10 public holidays and 5 sick days."
        },
        {
            "question": "What is the remote work policy?",
            "answer": "Employees can work remotely up to 3 days per week after completing their probation period. Full remote work requires manager approval."
        },
        {
            "question": "How do I apply for leave?",
            "answer": "Leave applications should be submitted through the HR portal at least 2 weeks in advance. Emergency leave can be approved by your direct manager."
        },
        {
            "question": "What health insurance benefits are provided?",
            "answer": "We offer comprehensive health insurance covering medical, dental, and vision care. Family coverage is available with a 50% company contribution."
        },
        {
            "question": "Is there a performance bonus?",
            "answer": "Yes, employees are eligible for annual performance bonuses ranging from 10-25% of base salary based on individual and company performance."
        },
        {
            "question": "What is the dress code?",
            "answer": "We have a business casual dress code. Jeans are acceptable on Fridays. Client-facing roles may require business formal attire."
        },
        {
            "question": "Are there professional development opportunities?",
            "answer": "Yes, we provide an annual learning budget of $2,000 per employee for courses, certifications, and conferences."
        },
        {
            "question": "What is the probation period?",
            "answer": "New employees have a 3-month probation period with a performance review at the end. Extensions may apply for certain roles."
        },
        {
            "question": "How does the 401k retirement plan work?",
            "answer": "We offer a 401k plan with up to 5% company matching. Employees are eligible after 6 months of employment."
        }
    ]
}

# Save sample dataset
with open('sample_hr_faqs.json', 'w') as f:
    json.dump(SAMPLE_DATASET, f, indent=2)

print("Sample dataset created: sample_hr_faqs.json")


Sample dataset created: sample_hr_faqs.json


# **STEP 4: DATA LOADING & PREPROCESSING**

In [12]:
class DataLoader:
    """Handles loading and preprocessing of various data formats"""

    @staticmethod
    def load_json(filepath: str) -> List[Dict]:
        """Load JSON file"""
        with open(filepath, 'r', encoding='utf-8') as f:
            data = json.load(f)

        # Handle nested structures
        if isinstance(data, dict):
            if 'faqs' in data:
                return data['faqs']
            return [data]
        return data

    @staticmethod
    def load_csv(filepath: str, text_column: str = None) -> List[Dict]:
        """Load CSV file"""
        df = pd.read_csv(filepath)

        # Auto-detect text column if not specified
        if text_column is None:
            text_column = df.columns[0]

        # Convert to list of dicts
        return df.to_dict('records')

    @staticmethod
    def load_txt(filepath: str, chunk_size: int = 500) -> List[Dict]:
        """Load text file and split into chunks"""
        with open(filepath, 'r', encoding='utf-8') as f:
            text = f.read()

        # Simple chunking by character count
        chunks = []
        for i in range(0, len(text), chunk_size):
            chunk = text[i:i+chunk_size]
            if chunk.strip():
                chunks.append({"text": chunk.strip()})

        return chunks

    @staticmethod
    def load_pdf(filepath: str) -> List[Dict]:
        """Load PDF file and extract text"""
        chunks = []
        with open(filepath, 'rb') as f:
            pdf_reader = PyPDF2.PdfReader(f)
            for page_num, page in enumerate(pdf_reader.pages):
                text = page.extract_text()
                if text.strip():
                    chunks.append({
                        "text": text.strip(),
                        "page": page_num + 1
                    })
        return chunks

    @staticmethod
    def preprocess_documents(documents: List[Dict]) -> Tuple[List[str], List[Dict]]:
        """
        Extract text from documents for embedding
        Returns: (texts_for_embedding, original_documents)
        """
        texts = []

        for doc in documents:
            # Handle different document structures
            if 'question' in doc and 'answer' in doc:
                # FAQ format: combine question and answer
                text = f"Q: {doc['question']}\nA: {doc['answer']}"
            elif 'text' in doc:
                text = doc['text']
            elif 'content' in doc:
                text = doc['content']
            else:
                # Fallback: concatenate all values
                text = ' '.join(str(v) for v in doc.values())

            texts.append(text)

        return texts, documents

print("Data loading utilities ready!")

Data loading utilities ready!


# **STEP 5: EMBEDDING MODEL & FINE-TUNING**

In [13]:
class EmbeddingModel:
    """Manages embedding model and fine-tuning"""

    def __init__(self, model_name: str = 'all-MiniLM-L6-v2'):
        """Initialize embedding model"""
        self.model_name = model_name
        self.model = SentenceTransformer(model_name)
        print(f"Loaded embedding model: {model_name}")

    def create_training_pairs(self, documents: List[Dict]) -> List[InputExample]:
        """
        Create training pairs for fine-tuning
        For FAQ data: question-answer pairs
        For general text: generate synthetic pairs
        """
        examples = []

        for doc in documents:
            if 'question' in doc and 'answer' in doc:
                # FAQ format: question is query, answer is positive
                examples.append(InputExample(
                    texts=[doc['question'], doc['answer']],
                    label=1.0  # Similar
                ))

                # Create negative pair with different answer
                for other_doc in documents:
                    if other_doc != doc:
                        examples.append(InputExample(
                            texts=[doc['question'], other_doc['answer']],
                            label=0.0  # Dissimilar
                        ))
                        break  # Only one negative pair per question

        return examples

    def fine_tune(self, train_examples: List[InputExample],
                  epochs: int = 3, batch_size: int = 16):
        """Fine-tune the embedding model"""
        print(f"🔧 Fine-tuning on {len(train_examples)} examples...")

        # Create dataloader - using TorchDataLoader to avoid conflict
        train_dataloader = TorchDataLoader(train_examples, shuffle=True, batch_size=batch_size)

        # Define loss function
        train_loss = losses.CosineSimilarityLoss(self.model)

        # Fine-tune
        self.model.fit(
            train_objectives=[(train_dataloader, train_loss)],
            epochs=epochs,
            warmup_steps=100,
            show_progress_bar=True
        )

        print("Fine-tuning completed!")

    def encode(self, texts: List[str]) -> np.ndarray:
        """Encode texts to embeddings"""
        return self.model.encode(texts, show_progress_bar=True)

    def save(self, path: str):
        """Save fine-tuned model"""
        self.model.save(path)
        print(f"Model saved to: {path}")

    def load(self, path: str):
        """Load fine-tuned model"""
        self.model = SentenceTransformer(path)
        print(f"Model loaded from: {path}")

print("Embedding model class ready!")

Embedding model class ready!


# **STEP 6: FAISS VECTOR STORE**

In [14]:
class FAISSVectorStore:
    """Manages FAISS index for vector similarity search"""

    def __init__(self, dimension: int = 384):
        """Initialize FAISS index"""
        self.dimension = dimension
        self.index = faiss.IndexFlatL2(dimension)
        self.documents = []
        print(f"FAISS index initialized (dim={dimension})")

    def add_documents(self, embeddings: np.ndarray, documents: List[Dict]):
        """Add documents and their embeddings to the index"""
        # Normalize embeddings for cosine similarity
        faiss.normalize_L2(embeddings)

        self.index.add(embeddings.astype('float32'))
        self.documents.extend(documents)

        print(f"Added {len(documents)} documents to index")
        print(f"Total documents in index: {len(self.documents)}")

    def search(self, query_embedding: np.ndarray, top_k: int = 3) -> List[Dict]:
        """Search for most similar documents"""
        # Normalize query embedding
        faiss.normalize_L2(query_embedding.reshape(1, -1))

        # Search
        distances, indices = self.index.search(
            query_embedding.reshape(1, -1).astype('float32'),
            top_k
        )

        # Retrieve documents
        results = []
        for idx, distance in zip(indices[0], distances[0]):
            if idx < len(self.documents):
                result = self.documents[idx].copy()
                result['similarity_score'] = float(1 / (1 + distance))  # Convert to similarity
                results.append(result)

        return results

    def save(self, index_path: str, docs_path: str):
        """Save FAISS index and documents"""
        faiss.write_index(self.index, index_path)

        with open(docs_path, 'wb') as f:
            pickle.dump(self.documents, f)

        print(f"Index saved to: {index_path}")
        print(f"Documents saved to: {docs_path}")

    def load(self, index_path: str, docs_path: str):
        """Load FAISS index and documents"""
        self.index = faiss.read_index(index_path)

        with open(docs_path, 'rb') as f:
            self.documents = pickle.load(f)

        print(f"Index loaded from: {index_path}")
        print(f"Loaded {len(self.documents)} documents")

print("FAISS vector store ready!")

FAISS vector store ready!


# **STEP 7: RAG PIPELINE**

In [15]:
class RAGPipeline:
    """Complete RAG pipeline with retrieval and generation"""

    def __init__(self, groq_api_key: str, model_name: str = "llama-3.1-8b-instant"):
        """Initialize RAG pipeline"""
        self.embedding_model = None
        self.vector_store = None
        self.groq_client = Groq(api_key=groq_api_key)
        self.model_name = model_name
        print(f"RAG Pipeline initialized with model: {model_name}")

    def build_index(self, documents: List[Dict], fine_tune: bool = True):
        """Build vector index from documents"""
        print("🏗️ Building vector index...")

        # Initialize embedding model
        self.embedding_model = EmbeddingModel()

        # Fine-tune if requested
        if fine_tune and len(documents) >= 2:
            train_examples = self.embedding_model.create_training_pairs(documents)
            if train_examples:
                self.embedding_model.fine_tune(train_examples, epochs=3)

        # Preprocess documents
        texts, processed_docs = DataLoader.preprocess_documents(documents)

        # Generate embeddings
        embeddings = self.embedding_model.encode(texts)

        # Create vector store
        self.vector_store = FAISSVectorStore(dimension=embeddings.shape[1])
        self.vector_store.add_documents(embeddings, processed_docs)

        print("Index built successfully!")

    def retrieve(self, query: str, top_k: int = 3) -> List[Dict]:
        """Retrieve relevant documents"""
        if self.vector_store is None:
            raise ValueError("Vector store not initialized. Call build_index first.")

        # Encode query
        query_embedding = self.embedding_model.encode([query])

        # Search
        results = self.vector_store.search(query_embedding, top_k)

        return results

    def generate(self, query: str, context: List[Dict], temperature: float = 0.7) -> str:
        """Generate answer using Groq API"""
        # Format context
        context_text = "\n\n".join([
            f"Document {i+1}:\n{self._format_document(doc)}"
            for i, doc in enumerate(context)
        ])

        # Create prompt
        prompt = f"""You are a helpful assistant. Use the following context to answer the question. If the answer cannot be found in the context, say so.

Context:
{context_text}

Question: {query}

Answer:"""

        # Call Groq API
        try:
            response = self.groq_client.chat.completions.create(
                messages=[{"role": "user", "content": prompt}],
                model=self.model_name,
                temperature=temperature,
                max_tokens=500
            )

            return response.choices[0].message.content

        except Exception as e:
            return f"Error generating response: {str(e)}"

    def query(self, query: str, top_k: int = 3, temperature: float = 0.7) -> Dict:
        """Complete RAG query: retrieve + generate"""
        # Retrieve
        retrieved_docs = self.retrieve(query, top_k)

        # Generate
        answer = self.generate(query, retrieved_docs, temperature)

        return {
            "query": query,
            "answer": answer,
            "retrieved_context": retrieved_docs,
            "num_retrieved": len(retrieved_docs)
        }

    @staticmethod
    def _format_document(doc: Dict) -> str:
        """Format document for display"""
        if 'question' in doc and 'answer' in doc:
            return f"Q: {doc['question']}\nA: {doc['answer']}"
        elif 'text' in doc:
            return doc['text']
        else:
            return str(doc)

    def save(self, model_path: str, index_path: str, docs_path: str):
        """Save entire pipeline"""
        if self.embedding_model:
            self.embedding_model.save(model_path)
        if self.vector_store:
            self.vector_store.save(index_path, docs_path)
        print("Pipeline saved!")

    def load(self, model_path: str, index_path: str, docs_path: str):
        """Load entire pipeline"""
        self.embedding_model = EmbeddingModel()
        self.embedding_model.load(model_path)

        # Get embedding dimension from model
        test_embedding = self.embedding_model.encode(["test"])
        self.vector_store = FAISSVectorStore(dimension=test_embedding.shape[1])
        self.vector_store.load(index_path, docs_path)

        print("Pipeline loaded!")

print("RAG Pipeline ready!")

RAG Pipeline ready!


# **STEP 8: EVALUATION METRICS**

In [16]:
class RAGEvaluator:
    """Evaluation metrics for RAG system"""

    @staticmethod
    def retrieval_accuracy(retrieved_docs: List[Dict],
                          ground_truth_doc: Dict) -> float:
        """Check if ground truth is in retrieved documents"""
        for doc in retrieved_docs:
            if doc == ground_truth_doc:
                return 1.0
        return 0.0

    @staticmethod
    def mean_reciprocal_rank(retrieved_docs: List[Dict],
                            ground_truth_doc: Dict) -> float:
        """Calculate MRR"""
        for i, doc in enumerate(retrieved_docs):
            if doc == ground_truth_doc:
                return 1.0 / (i + 1)
        return 0.0

    @staticmethod
    def average_similarity(query_embedding: np.ndarray,
                          doc_embeddings: List[np.ndarray]) -> float:
        """Calculate average cosine similarity"""
        similarities = []
        for doc_emb in doc_embeddings:
            sim = cosine_similarity(
                query_embedding.reshape(1, -1),
                doc_emb.reshape(1, -1)
            )[0][0]
            similarities.append(sim)

        return np.mean(similarities) if similarities else 0.0

print("Evaluation utilities ready!")

Evaluation utilities ready!


# **STEP 9: MAIN EXECUTION - BUILD PIPELINE**

In [17]:
print("\n" + "="*60)
print("BUILDING RAG PIPELINE WITH SAMPLE DATA")
print("="*60 + "\n")

# Configuration
GROQ_API_KEY = "YOUR_API_KEY"
MODEL_NAME = "llama-3.1-8b-instant"

# Load sample data
print("Loading sample dataset...")
documents = DataLoader.load_json('sample_hr_faqs.json')
print(f"Loaded {len(documents)} documents")

# Initialize pipeline
print("\nInitializing RAG Pipeline...")
rag = RAGPipeline(groq_api_key=GROQ_API_KEY, model_name=MODEL_NAME)

# Build index (with fine-tuning)
print("\nBuilding vector index with fine-tuning...")
rag.build_index(documents, fine_tune=True)

# Save pipeline
print("\nSaving pipeline...")
rag.save(
    model_path='fine_tuned_embeddings',
    index_path='faiss_index.bin',
    docs_path='documents.pkl'
)

# Test query
print("\n" + "="*60)
print("TESTING RAG PIPELINE")
print("="*60 + "\n")

test_query = "How many vacation days can I take?"
print(f"Query: {test_query}\n")

result = rag.query(test_query, top_k=3, temperature=0.7)

print(f"Answer:\n{result['answer']}\n")
print(f"Retrieved {result['num_retrieved']} relevant documents:")
for i, doc in enumerate(result['retrieved_context']):
    print(f"\n  Document {i+1} (Similarity: {doc.get('similarity_score', 0):.3f}):")
    print(f"  {rag._format_document(doc)[:200]}...")

print("\nPipeline test completed!")


BUILDING RAG PIPELINE WITH SAMPLE DATA

Loading sample dataset...
Loaded 10 documents

Initializing RAG Pipeline...
RAG Pipeline initialized with model: llama-3.1-8b-instant

Building vector index with fine-tuning...
🏗️ Building vector index...
Loaded embedding model: all-MiniLM-L6-v2
🔧 Fine-tuning on 20 examples...


wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: nareenasad07 (nareenasad07-lahore-college-for-women-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: Detected [groq] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai/
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


...

FAISS index initialized (dim=384)
Added 10 documents to index
Total documents in index: 10
Index built successfully!

Saving pipeline...


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Model saved to: fine_tuned_embeddings
Index saved to: faiss_index.bin
Documents saved to: documents.pkl
Pipeline saved!

TESTING RAG PIPELINE

Query: How many vacation days can I take?



Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Answer:
According to Document 1, full-time employees receive 20 vacation days per year.

Retrieved 3 relevant documents:

  Document 1 (Similarity: 0.572):
  Q: How many vacation days do employees get?
A: Full-time employees receive 20 vacation days per year, plus 10 public holidays and 5 sick days....

  Document 2 (Similarity: 0.410):
  Q: What is the remote work policy?
A: Employees can work remotely up to 3 days per week after completing their probation period. Full remote work requires manager approval....

  Document 3 (Similarity: 0.393):
  Q: What are the company's working hours?
A: Our standard working hours are 9:00 AM to 6:00 PM, Monday through Friday. We also offer flexible working arrangements....

Pipeline test completed!


# **ADDITIONAL TEST QUERIES**

In [34]:
# Test Query 2

test_query_2 = "What is the company's policy on working from home?"
print(f"Query 2: {test_query_2}\n")

result_2 = rag.query(test_query_2, top_k=3, temperature=0.7)

print(f"Answer:\n{result_2['answer']}\n")
print(f"Retrieved {result_2['num_retrieved']} relevant documents:")
for i, doc in enumerate(result_2['retrieved_context']):
    print(f"\n  Document {i+1} (Similarity: {doc.get('similarity_score', 0):.3f}):")
    print(f"  {rag._format_document(doc)[:200]}...")


Query 2: What is the company's policy on working from home?



Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Answer:
The company's policy on working from home is as follows:

- Employees can work remotely up to 3 days per week after completing their probation period.
- Full remote work requires manager approval.

Retrieved 3 relevant documents:

  Document 1 (Similarity: 0.489):
  Q: What is the remote work policy?
A: Employees can work remotely up to 3 days per week after completing their probation period. Full remote work requires manager approval....

  Document 2 (Similarity: 0.434):
  Q: What are the company's working hours?
A: Our standard working hours are 9:00 AM to 6:00 PM, Monday through Friday. We also offer flexible working arrangements....

  Document 3 (Similarity: 0.381):
  Q: How many vacation days do employees get?
A: Full-time employees receive 20 vacation days per year, plus 10 public holidays and 5 sick days....


In [35]:
test_query_3 = "Tell me about the health benefits and insurance coverage"
print(f"Query 3: {test_query_3}\n")

result_3 = rag.query(test_query_3, top_k=3, temperature=0.5)

print(f"Answer:\n{result_3['answer']}\n")
print(f"Retrieved {result_3['num_retrieved']} relevant documents:")
for i, doc in enumerate(result_3['retrieved_context']):
    print(f"\n  Document {i+1} (Similarity: {doc.get('similarity_score', 0):.3f}):")
    print(f"  {rag._format_document(doc)[:200]}...")


Query 3: Tell me about the health benefits and insurance coverage



Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Answer:
According to Document 1, the health insurance benefits provided are:

- Comprehensive health insurance
- Covering medical care
- Covering dental care
- Covering vision care
- Family coverage is available
- The company contributes 50% towards the family coverage.

Retrieved 3 relevant documents:

  Document 1 (Similarity: 0.609):
  Q: What health insurance benefits are provided?
A: We offer comprehensive health insurance covering medical, dental, and vision care. Family coverage is available with a 50% company contribution....

  Document 2 (Similarity: 0.390):
  Q: How does the 401k retirement plan work?
A: We offer a 401k plan with up to 5% company matching. Employees are eligible after 6 months of employment....

  Document 3 (Similarity: 0.386):
  Q: How many vacation days do employees get?
A: Full-time employees receive 20 vacation days per year, plus 10 public holidays and 5 sick days....
